# Self Query Retrieval

This notebook is organized into categories so that every block of code is easy to follow. The underlying code is unchanged from the original notebook. What has been added is structure, grouping of related cells, and explanations of what each part does and why it matters.

The notebook covers three connected ideas. First, a basic Retrieval Augmented Generation flow is shown together with the reason it can lose information. Second, a small metadata rich movie dataset is used to show exactly where plain similarity search struggles. Third, self query retrieval is introduced, where a language model turns a natural language question into a structured query that filters on metadata such as year, director, rating, and genre before handing the result to another language model for a final answer.


## Category 1: Understanding the Basic RAG Flow and Its Limits

Before writing any code, it helps to understand what problem this notebook is solving. This category groups the original conceptual notes about the basic flow and the issue that motivates self query retrieval.


### When to use the basic flow

This is the most basic flow, and it works well for documents such as PDFs where the data is linear and there is no major interdependency among different parts of the document.


### The issue with plain similarity search

Plain similarity search filters down to only the top k chunks that are semantically similar to the user question, but this has two weaknesses.

1. The returned chunk might not actually be the relevant one for the question being asked.
2. The search only looks at the words present in the query, without any awareness of how a chunk depends on other chunks. This can lead to a loss of information that the answer actually needed.

Self query retrieval, introduced later in this notebook, addresses part of this problem by letting a language model translate a natural language question into a structured filter over metadata fields, so the search is not limited to plain text similarity alone.


## Category 2: Environment Setup

This category installs the packages used throughout the notebook and loads the API key needed to call the language model. LangChain, OpenAI adjacent tooling, tiktoken, PyPDF2, and FAISS are installed first as general RAG dependencies, followed by the Groq integration, the community package, and the Chroma vector store integration used later for the metadata aware examples.


In [ ]:
!pip -q install langchain openai tiktoken PyPDF2 faiss-cpu

In [ ]:
!pip install -U langchain-groq

In [ ]:
!pip install -U langchain-community

In [ ]:
!pip install langchain_chroma

### An alternative embedding and chat model using Gemini

This commented out block is left as a reference. It shows how the notebook could instead use Google Generative AI embeddings and the Gemini chat model if a Gemini API key is preferred over Groq. It is not executed as part of the main flow below.


In [ ]:
####if you want to use gemini feel free to use this code.

'''
%pip install --upgrade --quiet  google-generativeai langchain-google-genai

import os
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

from langchain_google_genai import GoogleGenerativeAIEmbeddings
gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro")

result = llm.invoke("Write a ballad about LangChain")
print(result.content)
'''

### Loading the Groq API key

The Groq API key is stored in a local `.env` file and loaded with `python-dotenv`, so the language model calls in this notebook do not require the key to be hard coded anywhere.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

## Category 3: Building a Metadata Rich Vector Store

To demonstrate the difference between plain similarity search and self query retrieval, this category builds a small movie dataset where every entry carries useful metadata such as year, director, rating, and genre alongside its text description. This metadata is what a self query retriever will later be able to filter on.


### Importing the vector store and embedding components

`Chroma` is the vector database used to store and search the movie documents. `Document` is the LangChain data structure that pairs page content with a metadata dictionary. `HuggingFaceEmbeddings` is used to turn each movie description into a dense vector.


In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

### Creating the movie documents

Each `Document` below is a short movie description together with a metadata dictionary that records details such as year, director, rating, and genre. Notice that not every document has every field, which mirrors how real metadata is often incomplete. This dataset is intentionally small and easy to read so that the effect of filtering by metadata can be checked by eye.


In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="A hacker discovers reality is a simulation and leads a rebellion against the machines controlling it.",
        metadata={"year": 1999, "director": "Lana Wachowski, Lilly Wachowski", "rating": 8.7, "genre": "science fiction"},
    ),
    Document(
        page_content="A young lion prince flees his kingdom only to learn the true meaning of responsibility and bravery.",
        metadata={"year": 1994, "rating": 8.5, "genre": "animated"},
    ),
    Document(
        page_content="Batman faces off against the Joker, a criminal mastermind who plunges Gotham into chaos.",
        metadata={"year": 2008, "director": "Christopher Nolan", "rating": 9.0, "genre": "action"},
    ),
    Document(
        page_content="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.",
        metadata={"year": 2014, "director": "Christopher Nolan", "rating": 8.6, "genre": "science fiction"},
    ),
    Document(
        page_content="A hobbit sets out on an epic journey to destroy a powerful ring and save Middle-earth.",
        metadata={"year": 2001, "director": "Peter Jackson", "rating": 8.9, "genre": "fantasy"},
    ),
    Document(
        page_content="A young wizard discovers his magical destiny while battling the dark wizard threatening the world.",
        metadata={"year": 2001, "director": "Chris Columbus", "rating": 7.6, "genre": "fantasy"},
    ),
    Document(
        page_content="An unlikely fellowship battles evil forces to protect a magical ring from falling into the wrong hands.",
        metadata={"year": 2002, "director": "Peter Jackson", "rating": 8.8, "genre": "fantasy"},
    ),
    Document(
        page_content="A brave princess joins forces with an ice harvester and a snowman to save her kingdom from an endless winter.",
        metadata={"year": 2013, "director": "Chris Buck, Jennifer Lee", "rating": 7.4, "genre": "animated"},
    ),
    Document(
        page_content="A clownfish crosses the ocean with a forgetful blue tang to rescue his missing son.",
        metadata={"year": 2003, "director": "Andrew Stanton", "rating": 8.2, "genre": "animated"},
    ),
    Document(
        page_content="A lonely robot cleans an abandoned Earth and unexpectedly falls in love with another robot.",
        metadata={"year": 2008, "director": "Andrew Stanton", "rating": 8.4, "genre": "animated"},
    ),
    Document(
        page_content="A poor young man wins a fortune on a quiz show while recounting his extraordinary life story.",
        metadata={"year": 2008, "director": "Danny Boyle", "rating": 8.0, "genre": "drama"},
    ),
    Document(
        page_content="A brilliant mathematician struggles with mental illness while making groundbreaking discoveries.",
        metadata={"year": 2001, "director": "Ron Howard", "rating": 8.2, "genre": "biography"},
    ),
    Document(
        page_content="A genius billionaire builds an advanced suit of armor and becomes a superhero.",
        metadata={"year": 2008, "director": "Jon Favreau", "rating": 7.9, "genre": "action"},
    ),
    Document(
        page_content="Earth's mightiest heroes unite to stop an alien invasion led by Loki.",
        metadata={"year": 2012, "director": "Joss Whedon", "rating": 8.0, "genre": "superhero"},
    ),
    Document(
        page_content="A former Roman general seeks revenge against the corrupt emperor who murdered his family.",
        metadata={"year": 2000, "director": "Ridley Scott", "rating": 8.5, "genre": "historical drama"},
    ),
    Document(
        page_content="A boxer from Philadelphia gets an unexpected chance to fight the heavyweight champion.",
        metadata={"year": 1976, "director": "John G. Avildsen", "rating": 8.1, "genre": "sports drama"},
    ),
    Document(
        page_content="A determined FBI trainee works with a brilliant imprisoned cannibal to catch a serial killer.",
        metadata={"year": 1991, "director": "Jonathan Demme", "rating": 8.6, "genre": "thriller"},
    ),
    Document(
        page_content="A banker wrongly convicted of murder forms an unlikely friendship while planning an impossible escape.",
        metadata={"year": 1994, "director": "Frank Darabont", "rating": 9.3, "genre": "drama"},
    ),
    Document(
        page_content="A crime boss's youngest son is drawn into the dangerous world of organized crime.",
        metadata={"year": 1972, "director": "Francis Ford Coppola", "rating": 9.2, "genre": "crime"},
    ),
    Document(
        page_content="A serial killer uses the seven deadly sins as inspiration for a series of gruesome murders.",
        metadata={"year": 1995, "director": "David Fincher", "rating": 8.6, "genre": "crime thriller"},
    ),
    Document(
        page_content="A skilled race car driver learns humility and teamwork while competing in the Piston Cup.",
        metadata={"year": 2006, "director": "John Lasseter", "rating": 7.2, "genre": "animated"},
    ),
    Document(
        page_content="A socially awkward ogre embarks on a quest to rescue a princess and discovers true friendship.",
        metadata={"year": 2001, "director": "Andrew Adamson, Vicky Jenson", "rating": 7.9, "genre": "animated"},
    ),
    Document(
        page_content="An astronaut becomes stranded alone on Mars and must use science and ingenuity to survive.",
        metadata={"year": 2015, "director": "Ridley Scott", "rating": 8.0, "genre": "science fiction"},
    ),
    Document(
        page_content="A family with superpowers comes together to stop a dangerous villain while balancing everyday life.",
        metadata={"year": 2004, "director": "Brad Bird", "rating": 8.0, "genre": "animated"},
    ),
]

### Building the vector store

`Chroma.from_documents` embeds every movie description and stores the resulting vectors together with their metadata in a Chroma collection, ready to be searched.


In [ ]:
vectorstore = Chroma.from_documents(docs, embeddings)

## Category 4: Seeing the Limits of Plain Similarity Search

This category runs three questions directly against the vector store using plain similarity search, with no metadata filtering involved. Each question is designed to probe a different weakness. The first question asks about a specific year and rating combination, which plain similarity search has no reliable way to enforce. The second question asks for an entire category of movies, which similarity search can only approximate. The third question asks about a movie that does not actually exist in the dataset, showing that similarity search will still confidently return something even when nothing relevant is present.


### Question about a specific year and rating

Even though the year and the rating are explicit numeric facts in the metadata, plain similarity search can only match on text, so there is no guarantee the single correct answer is returned first or at all.


In [ ]:
question1 = "Which 1994 animated movie has a rating of 8.5?"

In [ ]:
vectorstore.similarity_search(question1)

### Question asking for an entire genre

Asking for every science fiction movie is a request for a category, not a single best match, which is a poor fit for a method that ranks documents by similarity score alone.


In [ ]:
question2 = "Show all science fiction movies."

In [ ]:
vectorstore.similarity_search(question2)

### Question about a movie that is not in the dataset

The Matrix does not appear in this dataset, even though a very similar plot summary about a hacker discovering a simulated reality does. This question checks what similarity search returns when there is no exact match to find.


In [ ]:
question3 = "What genre is the movie 'The Matrix' and who directed it?"

In [ ]:
vectorstore.similarity_search(question3)

### Creating a plain retriever and testing a range based question

A standard retriever is created from the vector store, returning the top three results by similarity. The query below asks for movies with a rating between two specific numbers, which is exactly the kind of numeric range condition that plain similarity search cannot enforce, since it has no concept of numeric comparison, only semantic closeness of text.


In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
query = "Show movies with rating between 8.0 and 8.5"
result = retriever.invoke(query)
result

In [ ]:
result[0].page_content

## Category 5: A Basic RAG Chain on Top of Plain Retrieval

This category wires the plain retriever from Category 4 into a full question answering chain, so a natural language question is answered end to end. This will serve as the baseline result to compare against once self query retrieval is introduced later.


In [ ]:
from langchain_groq import ChatGroq

from operator import itemgetter
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough


### Creating the chat model

`ChatGroq` is configured to use the `llama-3.3-70b-versatile` model with temperature set to zero, so its answers stay as consistent and factual as possible.


In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
)

### A helper for readable output

Long single line answers from the language model can be hard to read in a notebook. This helper wraps the text to a fixed width while preserving existing newlines, which is used whenever a final answer is printed later in the notebook.


In [ ]:
import textwrap
def wrap_text(text, width=90): #preserve_newlines
    # Split the input text into lines based on newline characters
    lines = text.split('\n')

    # Wrap each line individually
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]

    # Join the wrapped lines back together using newline characters
    wrapped_text = '\n'.join(wrapped_lines)

    return wrapped_text

### Defining the answer prompt and assembling the chain

The prompt instructs the model to answer strictly based on the retrieved context. The chain below wires together retrieval, prompt formatting, the language model, and output parsing into a single callable pipeline, using `RunnablePassthrough` so that the original question flows through unchanged alongside the retrieved context.


In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### Running the baseline chain

Two questions are run through this baseline chain. The first asks for movies directed by a specific person, and the second asks for movies from a specific year. Since the underlying retriever only performs plain similarity search, its ability to answer these precisely still depends on how well the question's wording happens to match the stored text.


In [ ]:
question= "Show movies directed by Christopher Nolan"

In [ ]:
text_reply = chain.invoke(question)

In [ ]:
print(wrap_text(text_reply))

In [ ]:
question2 = "Find movies released in 2008"

In [ ]:
text_reply = chain.invoke(question2)
text_reply

## Category 6: Introducing Self Query Retrieval

This category explains and then builds a self query retriever, the main technique this notebook is building toward. A self query retriever is a retrieval system that can turn a natural language question into a structured query, so the search can filter on metadata fields directly instead of relying purely on text similarity.

The process works in four steps.

1. The person provides a question in plain English.
2. A language model is used to understand the intent and meaning behind that question.
3. The language model translates the question into a structured query that a search engine can understand, including keywords and filters based on the details mentioned in the question.
4. That structured query is used to search the underlying vector store, and the most relevant documents are returned.

This technique is especially useful when the answer lives in a small subset of a much larger collection. For example, if every chunk of a large document carries a metadata field such as department name, then a question about one specific department can be answered by filtering on that metadata field rather than relying on wording alone. Overall, self query retrieval combines the language understanding of a large language model with structured metadata filtering, giving a more precise and user centered approach to retrieval than plain similarity search on its own.


### Installing the remaining packages for self query retrieval

A few additional packages are installed here, some of which duplicate earlier installs, reflecting how the notebook was originally built up interactively. `langchain_chroma` is reinstalled since it is central to this next section, and `lark` is installed further below because the structured query parser depends on it to parse the query language produced by the language model.


In [ ]:
!pip install langchain

In [ ]:
%pip install --upgrade --quiet langchain-chroma

In [ ]:
!pip install -U langchain-groq    # for openai !pip install langchain_openai

In [ ]:
!pip install langchain_chroma

### Rebuilding the embeddings, documents, and vector store

The same imports, embedding model, movie documents, and Chroma vector store from Category 3 are recreated here. This duplication mirrors the structure of the original notebook, where this section was written to be runnable on its own, and it is kept unchanged so the code continues to match the original exactly.


In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="A hacker discovers reality is a simulation and leads a rebellion against the machines controlling it.",
        metadata={"year": 1999, "director": "Lana Wachowski, Lilly Wachowski", "rating": 8.7, "genre": "science fiction"},
    ),
    Document(
        page_content="A young lion prince flees his kingdom only to learn the true meaning of responsibility and bravery.",
        metadata={"year": 1994, "rating": 8.5, "genre": "animated"},
    ),
    Document(
        page_content="Batman faces off against the Joker, a criminal mastermind who plunges Gotham into chaos.",
        metadata={"year": 2008, "director": "Christopher Nolan", "rating": 9.0, "genre": "action"},
    ),
    Document(
        page_content="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.",
        metadata={"year": 2014, "director": "Christopher Nolan", "rating": 8.6, "genre": "science fiction"},
    ),
    Document(
        page_content="A hobbit sets out on an epic journey to destroy a powerful ring and save Middle-earth.",
        metadata={"year": 2001, "director": "Peter Jackson", "rating": 8.9, "genre": "fantasy"},
    ),
    Document(
        page_content="A young wizard discovers his magical destiny while battling the dark wizard threatening the world.",
        metadata={"year": 2001, "director": "Chris Columbus", "rating": 7.6, "genre": "fantasy"},
    ),
    Document(
        page_content="An unlikely fellowship battles evil forces to protect a magical ring from falling into the wrong hands.",
        metadata={"year": 2002, "director": "Peter Jackson", "rating": 8.8, "genre": "fantasy"},
    ),
    Document(
        page_content="A brave princess joins forces with an ice harvester and a snowman to save her kingdom from an endless winter.",
        metadata={"year": 2013, "director": "Chris Buck, Jennifer Lee", "rating": 7.4, "genre": "animated"},
    ),
    Document(
        page_content="A clownfish crosses the ocean with a forgetful blue tang to rescue his missing son.",
        metadata={"year": 2003, "director": "Andrew Stanton", "rating": 8.2, "genre": "animated"},
    ),
    Document(
        page_content="A lonely robot cleans an abandoned Earth and unexpectedly falls in love with another robot.",
        metadata={"year": 2008, "director": "Andrew Stanton", "rating": 8.4, "genre": "animated"},
    ),
    Document(
        page_content="A poor young man wins a fortune on a quiz show while recounting his extraordinary life story.",
        metadata={"year": 2008, "director": "Danny Boyle", "rating": 8.0, "genre": "drama"},
    ),
    Document(
        page_content="A brilliant mathematician struggles with mental illness while making groundbreaking discoveries.",
        metadata={"year": 2001, "director": "Ron Howard", "rating": 8.2, "genre": "biography"},
    ),
    Document(
        page_content="A genius billionaire builds an advanced suit of armor and becomes a superhero.",
        metadata={"year": 2008, "director": "Jon Favreau", "rating": 7.9, "genre": "action"},
    ),
    Document(
        page_content="Earth's mightiest heroes unite to stop an alien invasion led by Loki.",
        metadata={"year": 2012, "director": "Joss Whedon", "rating": 8.0, "genre": "superhero"},
    ),
    Document(
        page_content="A former Roman general seeks revenge against the corrupt emperor who murdered his family.",
        metadata={"year": 2000, "director": "Ridley Scott", "rating": 8.5, "genre": "historical drama"},
    ),
    Document(
        page_content="A boxer from Philadelphia gets an unexpected chance to fight the heavyweight champion.",
        metadata={"year": 1976, "director": "John G. Avildsen", "rating": 8.1, "genre": "sports drama"},
    ),
    Document(
        page_content="A determined FBI trainee works with a brilliant imprisoned cannibal to catch a serial killer.",
        metadata={"year": 1991, "director": "Jonathan Demme", "rating": 8.6, "genre": "thriller"},
    ),
    Document(
        page_content="A banker wrongly convicted of murder forms an unlikely friendship while planning an impossible escape.",
        metadata={"year": 1994, "director": "Frank Darabont", "rating": 9.3, "genre": "drama"},
    ),
    Document(
        page_content="A crime boss's youngest son is drawn into the dangerous world of organized crime.",
        metadata={"year": 1972, "director": "Francis Ford Coppola", "rating": 9.2, "genre": "crime"},
    ),
    Document(
        page_content="A serial killer uses the seven deadly sins as inspiration for a series of gruesome murders.",
        metadata={"year": 1995, "director": "David Fincher", "rating": 8.6, "genre": "crime thriller"},
    ),
    Document(
        page_content="A skilled race car driver learns humility and teamwork while competing in the Piston Cup.",
        metadata={"year": 2006, "director": "John Lasseter", "rating": 7.2, "genre": "animated"},
    ),
    Document(
        page_content="A socially awkward ogre embarks on a quest to rescue a princess and discovers true friendship.",
        metadata={"year": 2001, "director": "Andrew Adamson, Vicky Jenson", "rating": 7.9, "genre": "animated"},
    ),
    Document(
        page_content="An astronaut becomes stranded alone on Mars and must use science and ingenuity to survive.",
        metadata={"year": 2015, "director": "Ridley Scott", "rating": 8.0, "genre": "science fiction"},
    ),
    Document(
        page_content="A family with superpowers comes together to stop a dangerous villain while balancing everyday life.",
        metadata={"year": 2004, "director": "Brad Bird", "rating": 8.0, "genre": "animated"},
    ),
]

In [ ]:
vectorstore = Chroma.from_documents(docs, embeddings)

### Importing the self query building blocks

`AttributeInfo` is used to describe each metadata field to the language model. `SelfQueryRetriever` is the retriever class that ties the query constructor, the vector store, and the metadata filter translator together. `ChatGroq` is imported again here to keep this section self contained.


In [ ]:
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain_groq import ChatGroq

### Describing the metadata fields

Each `AttributeInfo` entry tells the language model the name, a plain language description, and the data type of one metadata field. This description is what allows the language model to correctly turn a phrase like after 2010 into a numeric comparison on the year field, rather than treating it as ordinary search text.


In [ ]:
from langchain.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The director of the movie",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="The IMDb rating of the movie on a scale from 0 to 10",
        type="float",
    ),
    AttributeInfo(
        name="genre",
        description="The genre of the movie",
        type="string",
    ),
]

### Describing the documents themselves

Alongside the metadata field descriptions, the language model also needs a short description of what the document content itself represents, so it knows what the free text part of the query should search against.


In [ ]:
document_content_description = "Brief summary of a movie"

### Building the query constructor prompt

`get_query_constructor_prompt` takes the document description and the metadata field descriptions and builds a prompt template that instructs the language model on exactly how to turn a natural language question into a structured query object. Printing the prompt formatted with an example question shows the full instructions the language model receives, including the expected output format.


In [ ]:
from langchain.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

In [ ]:
prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
)

In [ ]:
prompt

In [ ]:
query = "Movies directed by Christopher Nolan after 2010"

print("=" * 80)
print(prompt.format(query=query))
print("=" * 80)

### Installing the query language parser

`lark` is the parsing library used by `StructuredQueryOutputParser` to turn the language model's text output into an actual structured query object with comparisons and filters.


In [ ]:
!pip install lark

### Assembling and testing the query constructor

The output parser, the prompt, and the language model are combined into a single `query_constructor` pipeline. Formatting the prompt with a placeholder question confirms the exact text sent to the model, and invoking the constructor with a real question shows the structured query it produces, for example turning a request for movies released in 2008 into an explicit equality comparison on the year field.


In [ ]:
output_parser = StructuredQueryOutputParser.from_components()

In [ ]:
query_constructor = prompt | llm | output_parser

In [ ]:
print(prompt.format(query="dummy question"))

In [ ]:
query_constructor.invoke(
    {
        "query": "Find movies released in 2008"
    }
)

### Reading the structured query result

The commented line below shows an example of what the query constructor returns for the 2008 question, a `StructuredQuery` object containing an equality comparison on the year field. This confirms that the language model correctly identified both the field to filter on and the operator to use.


In [ ]:
#StructuredQuery(query=' ', filter=Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='year', value=2008), limit=None)

### Wiring the self query retriever to the vector store

`ChromaTranslator` converts the structured query produced by the query constructor into the specific filter syntax that the Chroma vector store understands. `SelfQueryRetriever` combines the query constructor, the vector store, and this translator into a single retriever that can be called just like any other LangChain retriever.


In [ ]:
from langchain.retrievers.self_query.chroma import ChromaTranslator

retriever = SelfQueryRetriever(
    query_constructor=query_constructor,
    vectorstore=vectorstore,
    structured_query_translator=ChromaTranslator(),
)

In [ ]:
!pip install -U langchain-community

### Testing the self query retriever directly

Asking for movies with a rating above 9.0 is exactly the kind of numeric threshold question that defeated plain similarity search earlier in the notebook. With the self query retriever, this phrase is translated into an explicit greater than comparison on the rating field, so only documents that actually satisfy that numeric condition are returned.


In [ ]:
retriever.invoke(
    "Show movies with rating above 9.0."
)

## Category 7: A Full RAG Chain Using Self Query Retrieval

This final category connects the self query retriever to a language model in a complete question answering chain, mirroring the baseline chain from Category 5, but now backed by metadata aware retrieval instead of plain similarity search.


In [ ]:

from operator import itemgetter
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough


### Defining the answer prompt

The same style of prompt as before is used, instructing the model to answer strictly from the provided context. `PromptTemplate` is used here directly rather than `ChatPromptTemplate`, since this section only needs a single formatted string rather than a structured chat message list.


In [ ]:
from langchain_core.prompts import PromptTemplate

template = """Answer the question based only on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

In [ ]:
print(type(prompt))

### Assembling the self query RAG chain

This chain has the same shape as the baseline chain built earlier, but the `context` step now uses the self query retriever, so retrieval itself is aware of metadata filters such as rating thresholds before the language model ever sees the retrieved text.


In [ ]:
chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Running the self query RAG chain

Two questions are run through this chain. The first asks for movies with a rating greater than a specific number, and the second combines a director name with a rating threshold, a compound condition that plain similarity search has no reliable way to satisfy but that self query retrieval can express directly as a combined filter.


In [ ]:
text_reply = chain.invoke("Find movies with rating greater than 8.5.")

print(wrap_text(text_reply))


In [ ]:
text_reply = chain.invoke("Show Christopher Nolan movies with rating above 8.5.")

print(wrap_text(text_reply))


## Conclusion

This notebook worked through two connected ideas in Retrieval Augmented Generation.

The first part showed the basic RAG flow and its natural limitation. Plain similarity search returns the chunks whose text is closest to the query, but it has no way to understand explicit facts such as a year, a rating threshold, or a named category. Running direct similarity search questions against a small metadata rich movie dataset made this concrete: a question about a specific year and rating, a question asking for an entire genre, and a question about a movie that does not even exist in the dataset all returned results that looked plausible but were not reliably correct, simply because similarity search can only compare text meaning, not verify facts.

The second part introduced self query retrieval as a targeted fix for exactly this gap. By describing the dataset's metadata fields to a language model through `AttributeInfo`, and by using `get_query_constructor_prompt` together with `StructuredQueryOutputParser`, the notebook built a pipeline that turns a natural language question into an explicit structured query with comparisons such as equals and greater than. `SelfQueryRetriever`, combined with a vector store specific translator such as `ChromaTranslator`, then applies that structured query as a real metadata filter before any similarity search happens, and the results are noticeably more precise for the kinds of numeric and categorical questions that defeated plain retrieval earlier.

The overall takeaway is that plain similarity search and self query retrieval are complementary rather than competing techniques. Similarity search is fast and simple and works well when a question is really about the meaning of the text itself, while self query retrieval spends a little extra time asking a language model to understand the question first, so that any explicit facts in the question can be enforced as real filters. Choosing between them, or combining them as this notebook does by running similarity search inside the metadata filtered subset, depends on how much of the answer to a typical question is carried by structured facts versus by free text meaning.
